# Vision Embeddings — Full Encoder Benchmark

Tests every vision encoder on real images from The Cauldron,
benchmarks throughput, and compares baseline (sequential) vs optimized (pipelined) extraction.

In [ ]:
!pip install -q torch transformers datasets safetensors huggingface-hub tqdm accelerate pillow numpy
!pip install -q timm einops  # for V-JEPA 2.1

In [ ]:
!git clone https://github.com/gokayfem/vision-embeddings.git 2>/dev/null || (cd vision-embeddings && git pull)
%cd vision-embeddings
!pip install -q -e .

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"Torch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 1. Load test images from The Cauldron

In [ ]:
from datasets import load_dataset
from PIL import Image

NUM_TEST_IMAGES = 32

print(f"Streaming {NUM_TEST_IMAGES} images from HuggingFaceM4/the_cauldron (ai2d)...")
ds = load_dataset(
    "HuggingFaceM4/the_cauldron", name="ai2d",
    split="train", streaming=True,
)

test_images = []
for sample in ds:
    for img in (sample.get("images") or []):
        if img is not None:
            test_images.append(img.convert("RGB"))
        if len(test_images) >= NUM_TEST_IMAGES:
            break
    if len(test_images) >= NUM_TEST_IMAGES:
        break

print(f"Collected {len(test_images)} images")
for i in range(min(5, len(test_images))):
    print(f"  [{i}] size={test_images[i].size}")

## 2. Test every HF vision encoder

Loads each encoder, runs a correctness check, then benchmarks throughput.

Encoders are tested one at a time to avoid OOM — each is unloaded before the next.

In [ ]:
import gc
import time

import torch
from vision_embeddings import create_encoder, get_encoder_config, list_encoders
from vision_embeddings.auto_batch import find_optimal_batch_size

# skip torch_hub encoders in this cell (tested separately below)
HF_ENCODERS = [
    name for name in list_encoders()
    if get_encoder_config(name).loader != "torch_hub"
]

print(f"Testing {len(HF_ENCODERS)} HF encoders:\n")
results = {}

for name in HF_ENCODERS:
    cfg = get_encoder_config(name)
    print(f"{'='*60}")
    print(f"{name}")
    print(f"  model_id  : {cfg.model_id}")
    print(f"  loader    : {cfg.loader}")
    print(f"  embed_dim : {cfg.embed_dim}")
    print(f"  resolution: {cfg.resolution}")

    torch.cuda.empty_cache()
    gc.collect()

    try:
        encoder = create_encoder(name, device="cuda", compile_model=False)

        # correctness: single image
        emb = encoder.encode_batch([test_images[0]])
        print(f"  output    : {tuple(emb.shape)} {emb.dtype}")
        assert emb.ndim == 3
        assert emb.shape[2] == cfg.embed_dim
        actual_tokens = emb.shape[1]

        # find max batch size
        batch_size = find_optimal_batch_size(
            encoder, resolution=cfg.resolution, max_batch=64,
        )
        print(f"  auto batch: {batch_size}")

        # warmup
        for _ in range(2):
            encoder.encode_batch(test_images[:batch_size])
        torch.cuda.synchronize()

        # benchmark: encode all test images
        t0 = time.perf_counter()
        total_encoded = 0
        for start in range(0, len(test_images), batch_size):
            batch = test_images[start : start + batch_size]
            encoder.encode_batch(batch)
            total_encoded += len(batch)
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        throughput = total_encoded / elapsed
        ms_per_img = (elapsed / total_encoded) * 1000

        results[name] = {
            "model_id": cfg.model_id,
            "loader": cfg.loader,
            "embed_dim": cfg.embed_dim,
            "tokens": actual_tokens,
            "batch_size": batch_size,
            "img/s": round(throughput, 1),
            "ms/img": round(ms_per_img, 1),
            "status": "PASS",
        }

        print(f"  throughput: {throughput:.1f} img/s ({ms_per_img:.1f} ms/img)")
        print(f"  STATUS    : PASS")

    except Exception as exc:
        results[name] = {"status": "FAIL", "error": str(exc)}
        print(f"  STATUS    : FAIL — {exc}")

    finally:
        # unload
        try:
            del encoder
        except NameError:
            pass
        torch.cuda.empty_cache()
        gc.collect()

    print()

## 3. Test V-JEPA 2.1 (torch.hub) encoders

These load from `facebookresearch/vjepa2` via `torch.hub` and require `timm` + `einops`.

They are heavy (up to 2B params) — skip the larger ones if you only have a T4.

In [ ]:
TORCH_HUB_ENCODERS = [
    name for name in list_encoders()
    if get_encoder_config(name).loader == "torch_hub"
]

# On T4 (16GB), only test the smallest.
# Uncomment more if you have an A100.
TORCH_HUB_TO_TEST = [
    "vjepa2.1-vitb-384",
    # "vjepa2.1-vitl-384",
    # "vjepa2.1-vitg-384",
    # "vjepa2.1-vitG-384",
]

print(f"Available torch_hub encoders: {TORCH_HUB_ENCODERS}")
print(f"Testing: {TORCH_HUB_TO_TEST}\n")

for name in TORCH_HUB_TO_TEST:
    cfg = get_encoder_config(name)
    print(f"{'='*60}")
    print(f"{name}")
    print(f"  hub_name  : {cfg.hub_name}")
    print(f"  embed_dim : {cfg.embed_dim}")

    torch.cuda.empty_cache()
    gc.collect()

    try:
        encoder = create_encoder(name, device="cuda", compile_model=False)

        # V-JEPA encodes images as repeated-frame video clips
        # use batch_size=1 to be safe on T4
        emb = encoder.encode_batch([test_images[0]])
        print(f"  output    : {tuple(emb.shape)} {emb.dtype}")

        # benchmark with batch=1 (video models are VRAM-heavy)
        warmup_img = test_images[:1]
        for _ in range(2):
            encoder.encode_batch(warmup_img)
        torch.cuda.synchronize()

        n_bench = min(8, len(test_images))
        t0 = time.perf_counter()
        for i in range(n_bench):
            encoder.encode_batch([test_images[i]])
        torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0

        throughput = n_bench / elapsed
        ms_per_img = (elapsed / n_bench) * 1000

        results[name] = {
            "model_id": cfg.model_id,
            "loader": cfg.loader,
            "embed_dim": cfg.embed_dim,
            "tokens": emb.shape[1],
            "batch_size": 1,
            "img/s": round(throughput, 1),
            "ms/img": round(ms_per_img, 1),
            "status": "PASS",
        }

        print(f"  throughput: {throughput:.1f} img/s ({ms_per_img:.1f} ms/img)")
        print(f"  STATUS    : PASS")

    except Exception as exc:
        results[name] = {"status": "FAIL", "error": str(exc)}
        print(f"  STATUS    : FAIL — {exc}")

    finally:
        try:
            del encoder
        except NameError:
            pass
        torch.cuda.empty_cache()
        gc.collect()

    print()

## 4. Results summary

In [ ]:
print(f"{'Encoder':<26} {'Status':<6} {'Tokens':>6} {'Dim':>5} {'Batch':>5} {'img/s':>7} {'ms/img':>7}")
print("-" * 76)

passed = 0
failed = 0
for name, r in sorted(results.items()):
    status = r["status"]
    if status == "PASS":
        passed += 1
        print(
            f"{name:<26} {status:<6} "
            f"{r['tokens']:>6} {r['embed_dim']:>5} {r['batch_size']:>5} "
            f"{r['img/s']:>7} {r['ms/img']:>7}"
        )
    else:
        failed += 1
        err = r.get('error', 'unknown')[:50]
        print(f"{name:<26} {status:<6} {err}")

print("-" * 76)
print(f"{passed} passed, {failed} failed out of {len(results)}")

## 5. Baseline vs Optimized pipeline benchmark

Compares two approaches on the same encoder + 32 real images:

- **Baseline**: sequential encode -> blocking save -> blocking upload (mock)
- **Optimized**: parallel preprocessing + CUDA stream + background upload (mock)

Upload is mocked so we measure pure pipeline overhead, not network.

In [ ]:
import json
import tempfile
from pathlib import Path
from unittest.mock import MagicMock, patch
from concurrent.futures import ThreadPoolExecutor

from safetensors.torch import save_file

BENCH_ENCODER = "siglip2-so400m-384"  # change if not available
BENCH_IMAGES = 32
BENCH_SHARD = 16
BENCH_BATCH = 8

print(f"Loading {BENCH_ENCODER} for pipeline benchmark...")
bench_cfg = get_encoder_config(BENCH_ENCODER)
bench_enc = create_encoder(BENCH_ENCODER, device="cuda", compile_model=False)

# warmup
for _ in range(3):
    bench_enc.encode_batch(test_images[:BENCH_BATCH])
torch.cuda.synchronize()

bench_imgs = test_images[:BENCH_IMAGES]
print(f"Benchmarking with {len(bench_imgs)} images, batch={BENCH_BATCH}, shard={BENCH_SHARD}\n")

In [ ]:
# ---- BASELINE: sequential encode + blocking save (no background upload) ----

def baseline_pipeline(encoder, images, batch_size, shard_size, output_dir):
    """Simulates the original sequential pipeline (no threading, no streams)."""
    out = Path(output_dir) / "shards"
    out.mkdir(parents=True, exist_ok=True)
    shard_embs = []
    shard_meta = []
    shard_idx = 0
    total = 0

    for start in range(0, len(images), batch_size):
        batch = images[start : start + batch_size]

        # sequential preprocess + encode (no parallel prep, no stream)
        inputs = encoder.processor(images=batch, return_tensors="pt")
        pv = inputs["pixel_values"].to(encoder.device, dtype=encoder.dtype)
        with torch.no_grad():
            emb = encoder.model(pixel_values=pv).last_hidden_state.to(torch.float16).cpu()
        torch.cuda.synchronize()

        shard_embs.append(emb)
        shard_meta.extend([{"idx": start + i} for i in range(len(batch))])
        total += len(batch)

        # blocking save when shard full
        count = sum(e.shape[0] for e in shard_embs)
        if count >= shard_size:
            all_emb = torch.cat(shard_embs, dim=0)
            t_path = out / f"shard_{shard_idx:06d}.safetensors"
            m_path = out / f"shard_{shard_idx:06d}.json"
            save_file({"embeddings": all_emb[:shard_size]}, str(t_path))
            m_path.write_text(json.dumps(shard_meta[:shard_size]))
            # simulate blocking upload
            _ = t_path.read_bytes()
            left = all_emb.shape[0] - shard_size
            if left > 0:
                shard_embs = [all_emb[shard_size:]]
                shard_meta = shard_meta[shard_size:]
            else:
                shard_embs, shard_meta = [], []
            shard_idx += 1

    # flush last
    if shard_embs:
        all_emb = torch.cat(shard_embs, dim=0)
        t_path = out / f"shard_{shard_idx:06d}.safetensors"
        m_path = out / f"shard_{shard_idx:06d}.json"
        save_file({"embeddings": all_emb}, str(t_path))
        m_path.write_text(json.dumps(shard_meta))
        shard_idx += 1

    return total, shard_idx


# run baseline
torch.cuda.synchronize()
with tempfile.TemporaryDirectory() as tmpdir:
    t0 = time.perf_counter()
    total_b, shards_b = baseline_pipeline(
        bench_enc, bench_imgs, BENCH_BATCH, BENCH_SHARD, tmpdir,
    )
    torch.cuda.synchronize()
    baseline_time = time.perf_counter() - t0

baseline_throughput = total_b / baseline_time
print(f"BASELINE: {total_b} imgs, {shards_b} shards in {baseline_time:.2f}s")
print(f"  {baseline_throughput:.1f} img/s  ({baseline_time/total_b*1000:.1f} ms/img)")

In [ ]:
# ---- OPTIMIZED: pipelined encode + background save + parallel prep ----

from vision_embeddings.config import DatasetConfig
from vision_embeddings.pipeline import process_dataset

# We feed our pre-loaded images through the pipeline by patching load_dataset
# to return them as a fake streaming dataset.

fake_samples = [{"image": img} for img in bench_imgs]
ds_config = DatasetConfig(
    hf_id="fake/bench", image_column="image", split="train",
)

mock_api = MagicMock()
mock_api.list_repo_files.return_value = []

torch.cuda.synchronize()
with tempfile.TemporaryDirectory() as tmpdir:
    with patch("vision_embeddings.pipeline.load_dataset") as mock_ld, \
         patch("vision_embeddings.pipeline.HfApi", return_value=mock_api):
        mock_ld.return_value = iter(fake_samples)

        t0 = time.perf_counter()
        process_dataset(
            encoder=bench_enc,
            dataset_config=ds_config,
            encoder_config=bench_cfg,
            dataset_name="bench",
            repo_id="test/bench",
            output_dir=tmpdir,
            shard_size=BENCH_SHARD,
            batch_size=BENCH_BATCH,
            save_mode="tokens",
            delete_local=False,
        )
        torch.cuda.synchronize()
        optimized_time = time.perf_counter() - t0

    shard_dir = Path(tmpdir) / "bench" / "shards"
    shards_o = len(list(shard_dir.glob("*.safetensors")))

optimized_throughput = len(bench_imgs) / optimized_time
print(f"OPTIMIZED: {len(bench_imgs)} imgs, {shards_o} shards in {optimized_time:.2f}s")
print(f"  {optimized_throughput:.1f} img/s  ({optimized_time/len(bench_imgs)*1000:.1f} ms/img)")

In [ ]:
# ---- COMPARISON ----

speedup = baseline_time / optimized_time if optimized_time > 0 else float('inf')

print(f"\n{'='*60}")
print(f"PIPELINE BENCHMARK RESULTS ({BENCH_ENCODER})")
print(f"{'='*60}")
print(f"{'Images':<20} {len(bench_imgs)}")
print(f"{'Batch size':<20} {BENCH_BATCH}")
print(f"{'Shard size':<20} {BENCH_SHARD}")
print(f"{'='*60}")
print(f"{'Metric':<20} {'Baseline':>12} {'Optimized':>12} {'Speedup':>10}")
print(f"{'-'*60}")
print(f"{'Time (s)':<20} {baseline_time:>12.2f} {optimized_time:>12.2f} {speedup:>9.2f}x")
print(f"{'Throughput (img/s)':<20} {baseline_throughput:>12.1f} {optimized_throughput:>12.1f}")
print(f"{'Latency (ms/img)':<20} {baseline_time/total_b*1000:>12.1f} {optimized_time/len(bench_imgs)*1000:>12.1f}")
print(f"{'='*60}")
print(f"\nOptimized pipeline is {speedup:.2f}x faster than baseline.")
print(f"\nOptimizations applied:")
print(f"  - Background shard upload (non-blocking I/O)")
print(f"  - Parallel image preprocessing (ThreadPool)")
print(f"  - CUDA stream + pinned memory + non-blocking H2D")
print(f"  - Batched HF commits (create_commit vs upload_file)")
print(f"  - Auto batch-size tuning")

## 6. Save mode comparison (tokens vs pooled)

In [ ]:
import os

for mode in ("tokens", "pooled", "both"):
    fake_samples_mode = [{"image": img} for img in bench_imgs[:8]]
    mock_api_mode = MagicMock()
    mock_api_mode.list_repo_files.return_value = []

    with tempfile.TemporaryDirectory() as tmpdir:
        with patch("vision_embeddings.pipeline.load_dataset") as mock_ld, \
             patch("vision_embeddings.pipeline.HfApi", return_value=mock_api_mode):
            mock_ld.return_value = iter(fake_samples_mode)
            process_dataset(
                encoder=bench_enc,
                dataset_config=ds_config,
                encoder_config=bench_cfg,
                dataset_name=f"mode_{mode}",
                repo_id=f"test/mode_{mode}",
                output_dir=tmpdir,
                shard_size=8,
                batch_size=4,
                save_mode=mode,
                delete_local=False,
            )

        shard_dir = Path(tmpdir) / f"mode_{mode}" / "shards"
        for sf in sorted(shard_dir.glob("*.safetensors")):
            data = load_file(str(sf))
            total_bytes = sum(v.nelement() * v.element_size() for v in data.values())
            print(f"  [{mode:>7}] {sf.name}: ", end="")
            for k, v in data.items():
                print(f"{k}={tuple(v.shape)} ", end="")
            print(f"  ({total_bytes / 1024:.0f} KB)")

print("\nDone!")

## 7. Cleanup

In [ ]:
del bench_enc
torch.cuda.empty_cache()
gc.collect()
print(f"GPU memory freed: {torch.cuda.memory_allocated() / 1e6:.0f} MB allocated")